In [ ]:
!git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
!cd /content/ai-toolkit && pip install -r requirements.txt
!pip install -q diffusers accelerate

import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'
    AI_TOOLKIT = '/content/ai-toolkit'
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)
except ModuleNotFoundError:
    # Local run: find project root by looking for scripts/ directory
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    AI_TOOLKIT = str(Path(DRIVE_ROOT).parent / 'ai-toolkit')
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)

print(f"DRIVE_ROOT: {DRIVE_ROOT}")

In [ ]:
import json, shutil
from pathlib import Path

def prepare_aitoolkit_folder(jsonl_path: str, out_folder: str, max_items: int = None):
    """ai-toolkit expects folder with image.png + image.txt (caption) per sample."""
    out = Path(out_folder)
    out.mkdir(parents=True, exist_ok=True)
    with open(jsonl_path) as f:
        pairs = [json.loads(l) for l in f]
    if max_items:
        pairs = pairs[:max_items]
    for item in pairs:
        stem = Path(item['png_path']).stem
        shutil.copy(item['png_path'], out / f"{stem}.png")
        caption = item['caption']
        if not caption.startswith('LOGOIMG'):
            caption = f'LOGOIMG {caption}'
        (out / f"{stem}.txt").write_text(caption)
    print(f"Prepared {len(pairs)} pairs → {out_folder}")

prepare_aitoolkit_folder(
    f'{DRIVE_ROOT}/data/train_2k.jsonl',
    f'{DRIVE_ROOT}/data/train_2k_images',
)
prepare_aitoolkit_folder(
    f'{DRIVE_ROOT}/data/train_10k.jsonl',
    f'{DRIVE_ROOT}/data/train_10k_images',
)

In [ ]:
import subprocess
import yaml, tempfile, os

COLAB_ROOT = '/content/drive/MyDrive/liya_diplomCC'

def run_aitoolkit(config_path: str) -> int:
    """Run ai-toolkit with paths adapted for current environment."""
    with open(config_path) as f:
        content = f.read()
    # Substitute Colab paths with actual DRIVE_ROOT
    content = content.replace(COLAB_ROOT, DRIVE_ROOT)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        f.write(content)
        tmp = f.name
    try:
        result = subprocess.run(['python', f'{AI_TOOLKIT}/run.py', tmp])
        return result.returncode
    finally:
        os.unlink(tmp)

for rank in [4, 8, 16, 32]:
    config = f'{DRIVE_ROOT}/configs/sdxl_lora_r{rank}.yaml'
    print(f"\n{'='*50}\nTraining SDXL LoRA r={rank}...")
    rc = run_aitoolkit(config)
    print(f"r={rank}: {'DONE' if rc == 0 else 'FAILED'}")
    if rc != 0:
        raise RuntimeError(f"Training failed for r={rank}. Fix errors above before continuing.")

In [ ]:
import yaml, copy

# UPDATE BEST_RANK after reviewing Experiment 1 FID scores in notebook 07.
# Priority: lowest FID; use CLIP Score as tiebreaker.
BEST_RANK = 16

with open(f'{DRIVE_ROOT}/configs/sdxl_lora_r{BEST_RANK}.yaml') as f:
    base_cfg = yaml.safe_load(f)

for steps in [500, 1000, 4000]:  # 2000 already done in Exp.1
    cfg = copy.deepcopy(base_cfg)
    cfg['config']['name'] = f'sdxl_logo_lora_r{BEST_RANK}_s{steps}'
    cfg['config']['process'][0]['training_folder'] = (
        f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_RANK}_s{steps}'
    )
    cfg['config']['process'][0]['train']['steps'] = steps
    tmp = f'/tmp/sdxl_r{BEST_RANK}_s{steps}.yaml'
    with open(tmp, 'w') as f:
        yaml.dump(cfg, f)
    print(f"\nTraining r={BEST_RANK}, steps={steps}...")
    result = subprocess.run(['python', f'{AI_TOOLKIT}/run.py', tmp])
    print(f"r={BEST_RANK}, steps={steps}: {'DONE' if result.returncode == 0 else 'FAILED'}")
    if result.returncode != 0:
        raise RuntimeError(f"Training failed for steps={steps}. Fix errors above before continuing.")

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch
from pathlib import Path

TEST_PROMPTS = [
    "LOGOIMG minimalist coffee shop logo, circular, dark green, flat vector design",
    "LOGOIMG tech startup logo, geometric hexagon, blue gradient, sans-serif",
    "LOGOIMG bakery logo, wheat icon, warm brown, handcrafted artisan style",
    "LOGOIMG fitness brand, bold lion silhouette, orange and black, geometric",
    "LOGOIMG law firm, balanced scales, navy blue, serif elegant typography",
]

def generate_samples(lora_path: str, out_dir: str):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
    ).to("cuda")
    pipe.load_lora_weights(lora_path)
    pipe.set_progress_bar_config(disable=True)

    for i, prompt in enumerate(TEST_PROMPTS):
        imgs = pipe(
            prompt,
            negative_prompt="photorealistic, blurry, cluttered, complex background",
            num_images_per_prompt=2,
            generator=torch.Generator(device='cuda').manual_seed(42),
            guidance_scale=7.5,
            num_inference_steps=30,
            height=512, width=512,
        ).images
        for j, img in enumerate(imgs):
            img.save(f"{out_dir}/prompt{i:02d}_v{j}.png")

    del pipe
    torch.cuda.empty_cache()
    print(f"Saved samples → {out_dir}")

for rank in [4, 8, 16, 32]:
    generate_samples(
        f'{DRIVE_ROOT}/results/experiments/sdxl_r{rank}',
        f'{DRIVE_ROOT}/results/experiments/sdxl_r{rank}_samples',
    )

# Experiment 2: Generate samples for steps ablation variants
for steps in [500, 1000, 4000]:
    generate_samples(
        f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_RANK}_s{steps}',
        f'{DRIVE_ROOT}/results/experiments/sdxl_r{BEST_RANK}_s{steps}_samples',
    )
# Note: 2000-step baseline already in sdxl_r{BEST_RANK}_samples from Exp 1

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
ranks = [4, 8, 16, 32]

for row in range(5):
    for col, rank in enumerate(ranks):
        img_path = (f'{DRIVE_ROOT}/results/experiments/'
                    f'sdxl_r{rank}_samples/prompt{row:02d}_v0.png')
        if Path(img_path).exists():
            axes[row, col].imshow(Image.open(img_path))
        else:
            axes[row, col].text(0.5, 0.5, "Not yet\ngenerated",
                                ha='center', va='center', fontsize=8)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'LoRA r={rank}', fontsize=10, fontweight='bold')

plt.suptitle("SDXL LoRA — Rank Ablation (5 prompts × 4 ranks)", fontsize=13)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/rank_ablation_grid.png', dpi=150)
plt.show()